# Tutorial: Using MCP Tools with LangChain Agents

## Learning Objectives

By the end of this tutorial, you will be able to:
- Connect a LangGraph agent to an MCP server
- Use MCP tools within an agent workflow
- Understand the `MultiServerMCPClient` pattern

## Prerequisites

- Understanding of MCP concepts from the slides
- Familiarity with LangGraph agents and tool calling
- Basic async/await knowledge

## Setup

In [ ]:
# !pip install langchain-mcp-adapters fastmcp langgraph langchain-openai python-dotenv

In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "Please set OPENAI_API_KEY in your .env file"

print("Environment loaded!")

## The MCP Server

This directory contains `knowledge_base_server.py` - a FastMCP server with these tools:

| Tool | Description |
|------|-------------|
| `create_note` | Create a note with title, content, tags |
| `search_notes` | Search notes by keyword |
| `get_note` | Get a note by ID |
| `list_notes` | List all notes |
| `update_note` | Update a note |
| `delete_note` | Delete a note |
| `get_statistics` | Get KB statistics |

## Connecting to the MCP Server

Use `MultiServerMCPClient` to connect to MCP servers. For local servers, use `stdio` transport.

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

# Path to our MCP server
SERVER_PATH = Path(".").resolve() / "knowledge_base_server.py"

# MCP client configuration
mcp_config = {
    "knowledge_base": {
        "transport": "stdio",
        "command": sys.executable,
        "args": [str(SERVER_PATH)],
    }
}

# Initialize LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## Running the Agent with MCP Tools

The pattern:
1. Use `MultiServerMCPClient` as async context manager
2. Get tools with `client.get_tools()`
3. Pass tools to `create_react_agent()`

In [ ]:
async def run_agent(user_message: str):
    """Run a LangGraph agent with MCP tools."""
    async with MultiServerMCPClient(mcp_config) as client:
        tools = client.get_tools()
        
        print(f"Available tools: {[t.name for t in tools]}\n")
        
        agent = create_react_agent(llm, tools)
        result = await agent.ainvoke({
            "messages": [{"role": "user", "content": user_message}]
        })
        
        return result["messages"][-1].content

## Test: Creating Notes

In [ ]:
response = await run_agent(
    "Create a note titled 'Python Decorators' about how decorators work. "
    "Tag it with 'python' and 'tutorial'."
)
print(response)

In [ ]:
response = await run_agent(
    "Create a note about LangGraph basics covering nodes, edges, and state. "
    "Tag it with 'langgraph' and 'agents'."
)
print(response)

## Test: Searching and Listing

In [ ]:
response = await run_agent("Search for notes about Python.")
print(response)

In [ ]:
response = await run_agent("List all my notes and show me the knowledge base statistics.")
print(response)

## Test: Multi-Tool Task

In [ ]:
response = await run_agent(
    "Check what notes I have, then create a 'Learning Roadmap' note "
    "that references the topics from my existing notes. Tag it with 'meta'."
)
print(response)

## Inspecting Tool Calls

Let's see the full trace of what the agent does.

In [ ]:
async def run_agent_verbose(user_message: str):
    """Run agent and show detailed tool call trace."""
    async with MultiServerMCPClient(mcp_config) as client:
        tools = client.get_tools()
        agent = create_react_agent(llm, tools)
        
        result = await agent.ainvoke({
            "messages": [{"role": "user", "content": user_message}]
        })
        
        print("TRACE:")
        print("=" * 50)
        for i, msg in enumerate(result["messages"]):
            msg_type = type(msg).__name__
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f"[{i}] Tool: {tc['name']}({tc['args']})")
            elif msg_type == "ToolMessage":
                content = msg.content[:100] + "..." if len(msg.content) > 100 else msg.content
                print(f"[{i}] Result: {content}")
        print("=" * 50)
        
        return result["messages"][-1].content

In [ ]:
response = await run_agent_verbose(
    "Find notes tagged 'python' and add 'programming' to their tags."
)
print(f"\nFinal response:\n{response}")

## Cleanup

In [ ]:
import json

storage_file = Path(".").resolve() / "data" / "notes.json"
if storage_file.exists():
    storage_file.write_text(json.dumps({"notes": {}}, indent=2))
    print("Knowledge base cleared.")

## Summary

- Use `MultiServerMCPClient` with `stdio` transport for local MCP servers
- Call `client.get_tools()` to get LangChain-compatible tools
- Pass tools to `create_react_agent()` or any LangGraph agent
- The same MCP server works with Claude Desktop, Cursor, and custom agents